# Hybrid Search

Hybrid search combines KNN vector similarity with metadata filters in a single query. ApertureDB applies the filter **server-side during the search** — not as a post-filter — so you get the top-k most similar items that also satisfy the constraint.

This notebook shows three patterns:
1. KNN only
2. KNN + equality filter
3. KNN + range filter

## Connect to ApertureDB

**Option A: ApertureDB Cloud (recommended)**  
Sign up for a [free 30-day trial](https://cloud.aperturedata.io). Get your key from **Connect > Generate API Key**, add it to a `.env` file in this directory:
```
APERTUREDB_KEY=your_key_here
```

**Option B: Community Edition (local Docker)**  
Run this in a terminal before starting the notebook:
```bash
docker run -d --name aperturedb \
  -p 55555:55555 -e ADB_MASTER_KEY=admin -e ADB_FORCE_SSL=false \
  aperturedata/aperturedb-community
```

In [ ]:
%pip install --upgrade --quiet aperturedb python-dotenv sentence-transformers pandas

In [1]:
# Option A: ApertureDB Cloud
from dotenv import load_dotenv
load_dotenv()  # loads APERTUREDB_KEY from .env into the environment

True

In [ ]:
# Option B: Community Edition (local Docker)
# !adb config create localdb --active \
#     --host localhost --port 55555 \
#     --username admin --password admin \
#     --no-use-ssl --no-interactive

In [2]:
from aperturedb.CommonLibrary import create_connector

client = create_connector()
response, _ = client.query([{"GetStatus": {}}])
client.print_last_response()

[
    {
        "GetStatus": {
            "info": "OK",
            "status": 0,
            "system": "ApertureDB",
            "version": "0.19.6"
        }
    }
]


## Load and Ingest the Cookbook Dataset

We embed all 20 dishes with `all-MiniLM-L6-v2` and store them with cuisine and calorie metadata.

In [3]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer

dishes = pd.read_csv(
    "https://raw.githubusercontent.com/aperture-data/Cookbook/refs/heads/main/images.adb.csv"
)
dishes["description"] = dishes["dish_name"] + " - " + dishes["caption"]

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(dishes["description"].tolist(), normalize_embeddings=True)
print(f"Loaded {len(dishes)} dishes, embedding shape: {embeddings.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded 20 dishes, embedding shape: (20, 384)


In [4]:
SET_NAME = "cookbook_hybrid"

client.query([{
    "AddDescriptorSet": {
        "name": SET_NAME, "dimensions": 384, "engine": "HNSW", "metric": "CS",
    }
}])

# Add synthetic calorie counts for the range filter demo
import random
random.seed(42)

for i, row in dishes.iterrows():
    emb = embeddings[i].astype("float32")
    client.query([{
        "AddDescriptor": {
            "set": SET_NAME,
            "properties": {
                "dish_name": row["dish_name"],
                "cuisine":   row["food_tags"],
                "calories":  random.randint(300, 900),
            },
            "if_not_found": {"dish_name": ["==", row["dish_name"]]},
        }
    }], [emb.tobytes()])

print(f"Ingested {len(dishes)} descriptors")

Ingested 20 descriptors


## Pattern 1: KNN Only

Find the 3 dishes most similar to the query — no metadata filter.

In [5]:
query_text = "spicy curry with rice"
query_emb  = model.encode([query_text], normalize_embeddings=True)[0].astype("float32")

response, _ = client.query([{
    "FindDescriptor": {
        "set":         SET_NAME,
        "k_neighbors": 3,
        "distances":   True,
        "results":     {"all_properties": True},
    }
}], [query_emb.tobytes()])

print(f"KNN only — top 3 for '{query_text}':")
for e in response[0]["FindDescriptor"].get("entities", []):
    print(f"  {e['dish_name']:<30} [{e['cuisine']}]  score={1-e['_distance']:.3f}")

KNN only — top 3 for 'spicy curry with rice':
  rajma chawal                   [Indian]  score=0.416
  butter chicken with special fried rice and assorted naan breads [Indian]  score=0.447
  won ton soup, chicken chow mein, katsu chicken [Chinese]  score=0.538


## Pattern 2: KNN + Equality Filter

Restrict results to a specific cuisine. The filter is applied **during** the vector search, not after.

In [6]:
response, _ = client.query([{
    "FindDescriptor": {
        "set":         SET_NAME,
        "k_neighbors": 3,
        "constraints": {
            "cuisine": ["==", "Indian"],   # only search within Indian dishes
        },
        "distances":   True,
        "results":     {"all_properties": True},
    }
}], [query_emb.tobytes()])

print(f"KNN + cuisine==Indian — top 3 for '{query_text}':")
for e in response[0]["FindDescriptor"].get("entities", []):
    print(f"  {e['dish_name']:<30} [{e['cuisine']}]  score={1-e['_distance']:.3f}")

KNN + cuisine==Indian — top 3 for 'spicy curry with rice':
  rajma chawal                   [Indian]  score=0.416
  butter chicken with special fried rice and assorted naan breads [Indian]  score=0.447
  paneer bhurji                  [Indian]  score=0.580


## Pattern 3: KNN + Range Filter

Combine vector search with a numeric range constraint — for example, dishes under 600 calories.

In [7]:
response, _ = client.query([{
    "FindDescriptor": {
        "set":         SET_NAME,
        "k_neighbors": 5,
        "constraints": {
            "calories": ["<", 600],
        },
        "distances":   True,
        "results":     {"all_properties": True},
    }
}], [query_emb.tobytes()])

print(f"KNN + calories<600 — top 5 for '{query_text}':")
for e in response[0]["FindDescriptor"].get("entities", []):
    print(f"  {e['dish_name']:<30} calories={e['calories']}  score={1-e['_distance']:.3f}")

KNN + calories<600 — top 5 for 'spicy curry with rice':
  rajma chawal                   calories=414  score=0.416
  won ton soup, chicken chow mein, katsu chicken calories=332  score=0.538
  paneer bhurji                  calories=325  score=0.580
  negi miso ramen                calories=538  score=0.593
  vegetable tian with noodles    calories=395  score=0.614


## Cleanup

In [8]:
client.query([{"DeleteDescriptorSet": {"with_name": SET_NAME}}])
client.print_last_response()

[
    {
        "DeleteDescriptorSet": {
            "count": 1,
            "status": 0
        }
    }
]


## What's Next

- [Bulk Embeddings](https://docs.aperturedata.io/HowToGuides/start/BulkEmbeddings): ingest large numbers of embeddings with ParallelLoader
- [Building RAG Pipelines](https://docs.aperturedata.io/HowToGuides/Ingestion/RAG): use hybrid search as the retrieval step in a RAG chain
- [FindDescriptor reference](https://docs.aperturedata.io/query_language/Reference/descriptor_commands/desc_commands/FindDescriptor): full list of constraint operators